# 🎓 ImtiQan — Fine-Tuning Qwen3-1.7B with QLoRA

In [ ]:
from google.colab import drive

try:
    drive.mount('/content/drive', force_remount=True)
    print('Google Drive mounted!')
except Exception as e:
    print(f'Drive mount failed: {e}')
    print('Trying alternative...')
    drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ImtiQan', exist_ok=True)
os.makedirs('/content/drive/MyDrive/ImtiQan/checkpoints', exist_ok=True)
print('Folders created!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/ImtiQan'

if os.path.exists(REPO_DIR):
    print(f'Repo already exists at {REPO_DIR} — pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    print('Cloning repo...')
    GITHUB_URL = 'https://github.com/AhmedElbana22/Exam-generator-.git'
    !git clone {GITHUB_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f'\nWorking directory: {os.getcwd()}')
print('Files in repo:')
!ls

## Set Checkpoint Directory

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/ImtiQan/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

existing = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]

print(f'Checkpoint directory: {OUTPUT_DIR}')
if existing:
    print(f'Found existing checkpoints: {sorted(existing)}')
    print('Training will AUTO-RESUME from the latest one!')
else:
    print('No existing checkpoints — will start fresh')

In [ ]:
print('Installing libraries...')
!pip install -q transformers datasets peft accelerate bitsandbytes trl huggingface_hub
print('All libraries installed!')

import transformers, peft, trl, bitsandbytes
print(f'transformers : {transformers.__version__}')
print(f'peft         : {peft.__version__}')
print(f'trl          : {trl.__version__}')

In [ ]:
import os
from huggingface_hub import login
 
HF_TOKEN_Read = os.environ.get("HF_TOKEN_Read", "")

if not HF_TOKEN_Read:
    raise ValueError("HF_TOKEN_Read not found! Set it first.")

login(token=HF_TOKEN_Read)
print("Logged in to HuggingFace!")

login(token=HF_TOKEN_Read)
print('Logged in to HuggingFace!')

In [ ]:
import torch

print('=' * 55)
print(f'  PyTorch version  : {torch.__version__}')
print(f'  CUDA available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU name         : {name}')
    print(f'  VRAM             : {vram:.1f} GB')
    if vram >= 14:
        print('Enough VRAM for QLoRA fine-tuning!')
    else:
        print('Low VRAM — may run into memory errors')
else:
    print('NO GPU — Runtime → Change runtime type → T4 GPU')

## Load SQuAD v2 Dataset

130,000+ Q&A pairs from Wikipedia. We use 2,000 samples.
Increase TRAIN_SIZE to 5000–10000 for better quality.

In [ ]:
from datasets import load_dataset

print('Loading SQuAD v2 dataset...')
dataset = load_dataset('rajpurkar/squad_v2', split='train')
print(f'Full dataset    : {len(dataset):,} samples')

dataset = dataset.filter(lambda x: len(x['answers']['text']) > 0)
print(f'After filtering : {len(dataset):,} samples')

TRAIN_SIZE = 2000
dataset = dataset.select(range(TRAIN_SIZE))
print(f'Training subset : {len(dataset):,} samples')

print('\n--- Sample ---')
s = dataset[0]
print(f'Context  : {s["context"][:200]}...')
print(f'Question : {s["question"]}')
print(f'Answer   : {s["answers"]["text"][0]}')

## Format Dataset into Qwen3 Chat Format

Model learns: **text context → JSON {question, answer}**

In [ ]:
def format_sample(sample: dict) -> dict:
    context  = sample['context'][:500]
    question = sample['question'].replace('"', "'")
    answer   = sample['answers']['text'][0].replace('"', "'")

    text = (
        '<|im_start|>system\n'
        'You are an expert quiz generator. '
        'Generate a question and answer based on the given context. '
        'Always respond in valid JSON format only.\n'
        '<|im_end|>\n'
        '<|im_start|>user\n'
        'Generate a question and answer from this context:\n\n'
        f'{context}\n'
        '<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{{"question": "{question}", "answer": "{answer}"}}\n'
        '<|im_end|>'
    )
    return {'text': text}


print('Formatting dataset...')
formatted_dataset = dataset.map(
    format_sample,
    remove_columns=dataset.column_names,
    desc='Formatting',
)

print(f'Formatted {len(formatted_dataset):,} samples')
print('\n--- Formatted Sample ---')
print(formatted_dataset[0]['text'])

## Load Qwen3-1.7B with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen3-1.7B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded!')

print(f'\nLoading model with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

total = model.num_parameters()
print(f'\nModel loaded!')
print(f'Total parameters : {total:,}')
print(f'Approx VRAM used : ~{total * 0.5 / 1e9:.1f} GB (4-bit)')

## Apply LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

print('LoRA adapters applied!')
print()
model.print_trainable_parameters()

## Trainer config

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer
import os

OUTPUT_DIR = '/content/drive/MyDrive/ImtiQan/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    fp16=False,
    bf16=True,

    logging_steps=25,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=1,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    report_to='none',
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

print('Trainer ready!')
print(f'   Mixed precision  : bf16 (matches Qwen3)')
print(f'   Training samples : {len(tokenized_dataset):,}')
print(f'   Save every       : 50 steps → {OUTPUT_DIR}')

## Train

In [ ]:
import time, os

print('=' * 60)
print('Starting Fine-Tuning...')
print('=' * 60)

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith('checkpoint-')
    ]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getctime)
        print(f'Resuming from: {os.path.basename(last_checkpoint)}')
    else:
        print('Starting fresh')

print('💾 Checkpoints → Google Drive every 100 steps\n')

start = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
elapsed = (time.time() - start) / 60

print(f'\nTraining Complete! Time: {elapsed:.1f} minutes')

## Save Final Adapter to Drive

In [ ]:
import os

FINAL_DIR = '/content/drive/MyDrive/ImtiQan/final_adapter'
os.makedirs(FINAL_DIR, exist_ok=True)

print('Saving final LoRA adapter...')
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

files = os.listdir(FINAL_DIR)
size  = sum(os.path.getsize(os.path.join(FINAL_DIR, f)) for f in files) / 1e6

print(f'\nSaved to: {FINAL_DIR}')
print(f'   Files : {files}')
print(f'   Size  : {size:.1f} MB  (adapter only — base model stays on Hub)')

## Push to HuggingFace Hub

In [ ]:
YOUR_USERNAME = 'Elbana22'
REPO_NAME     = f'{YOUR_USERNAME}/imtiqan-qwen-1.7b-quiz-lora'

print(f'Pushing to: https://huggingface.co/{REPO_NAME}')

model.push_to_hub(REPO_NAME, private=False)
tokenizer.push_to_hub(REPO_NAME, private=False)

print(f'\n Done! → https://huggingface.co/{REPO_NAME}')
print(f'\nAdd to config.py:')
print(f'  self.FINE_TUNED_MODEL = "{REPO_NAME}"')

## Test the Fine-Tuned Model

In [ ]:
import json, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_PATH = '/content/drive/MyDrive/ImtiQan/final_adapter'

print('Loading base + adapter...')
base = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-1.7B',
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
ft_model.eval()
print('Loaded!')

test_contexts = [
    """The transformer architecture was introduced in 2017 in
    'Attention Is All You Need'. It uses self-attention to process
    sequences in parallel, replacing recurrent networks.""",
    """Photosynthesis converts sunlight, water, and CO2 into glucose
    and oxygen. It occurs in chloroplasts using chlorophyll.""",
]

def generate_question(ctx):
    prompt = (
        '<|im_start|>system\nYou are an expert quiz generator. '
        'Always respond in valid JSON only.\n<|im_end|>\n'
        '<|im_start|>user\nGenerate a question and answer from:\n\n'
        f'{ctx.strip()}\n<|im_end|>\n<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = ft_model.generate(
            **inputs, max_new_tokens=150, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

print('\n' + '=' * 60)
for i, ctx in enumerate(test_contexts, 1):
    print(f'\n Test {i}: {ctx.strip()[:80]}...')
    output = generate_question(ctx)
    print(f'Output: {output}')
    try:
        m = re.search(r'\{.*?\}', output, re.DOTALL)
        if m:
            p = json.loads(m.group())
            print(f' Q: {p.get("question")}')
            print(f'   A: {p.get("answer")}')
        else:
            print('  No JSON found')
    except:
        print('  JSON parse failed')
    print('-' * 60)

## Load SQuAD v2 Dataset

130,000+ Q&A pairs from Wikipedia. We use 2,000 samples.
Increase TRAIN_SIZE to 5000–10000 for better quality.

